Requirement:
We have colleced fire calls data file sf-fire-calls.csv.

 1. Read the data file
 2. Load it into a table for analysis
 3. Verify all 175296 records are loaded correctly
 4. The table is predefined as below

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_catalog.spark_db.sf_fire_calls (
  CallNumber INT, UnitID STRING, IncidentNumber INT, CallType STRING, CallDate DATE,
  WatchDate DATE, CallFinalDisposition STRING, AvailableDtTm TIMESTAMP, Address STRING,
  City STRING, Zipcode STRING, Battalion STRING, StationArea STRING, Box STRING,
  OriginalPriority STRING, Priority STRING, FinalPriority STRING, ALSUnit BOOLEAN,
  CallTypeGroup STRING, NumAlarms INT, UnitType STRING, UnitSequenceInCallDispatch INT,
  FirePreventionDistrict STRING, SupervisorDistrict STRING, Neighborhood STRING,
  Location STRING, RowID STRING, Delay DOUBLE);

Solution approach

1. Check the data file structure
2. Read the data file and create a dataframe

In [0]:
raw_fire_data_df=(
    spark.read.format("csv")
    .option("header","true")
    .option("infershema","true")
    .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/sf-fire-calls.csv")
)



3. Count the records(count)
4. Check the dataframe for potential problems(display/show)
5. Verify dataframe schema with the target table(print/schema)
6. Transform the dataframe to match target table structure(withColumns,to_data,to_timestamp,cast)

In [0]:
raw_fire_data_df.count()

In [0]:
raw_fire_data_df.display()

In [0]:
raw_fire_data_df.printSchema()

In [0]:
from pyspark.sql.functions import to_timestamp,to_date,expr,col
new_fire_df = (raw_fire_data_df
    .withColumn("AvailableDtTm",to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a"))
.withColumn("CallNumber",col("CallNumber").cast("INT"))
.withColumn("IncidentNumber",col("IncidentNumber").cast("INT"))
.withColumn("NumAlarms",col("NumAlarms").cast("INT"))
.withColumn("UnitSequenceInCallDispatch",col("UnitSequenceInCallDispatch").cast("INT"))
.withColumn("Delay",col("Delay").cast("DOUBLE"))
.withColumn("CallDate",to_date("CallDate","MM/dd/yyyy"))
.withColumn("WatchDate",to_date("WatchDate","MM/dd/yyyy"))
.withColumn("ALSUnit",col("ALSUnit").cast("BOOLEAN"))


# through this line I tried to add new cloumn to the dataframe
# .withColumn("FinalPriorityyy", col("FinalPriority").cast("string"))
)

# print structure(schema) of the dataframe 
new_fire_df.printSchema()
new_fire_df.display()



7. I will delete the newly added column named as FinalPriorityyy using DROP

I did the changes in dataframe but I deleted cell that is why nothing is available related to this here.

In [0]:
new_fire_df_2 = new_fire_df.drop("FinalPriorityyy")
new_fire_df_2.printSchema()


8. Save the final dataframe into target table

In [0]:
new_fire_df.write.mode("overwrite").saveAsTable("dev_catalog.spark_db.sf_fire_calls")

9. Verify the table count

In [0]:
%sql
select count(*) from dev_catalog.spark_db.sf_fire_calls